# Stage 5 — word intrusion

## What this is

Each topic has a list of top words (c-TF-IDF). A **good** topic's top words should hang together
(e.g. `pipeline`, `gas`, `capacity`, `nomination`, `mmbtu`).

**Word intrusion** is a human check: take the top 5 words of one topic, sneak in **one word from a different topic**
(the intruder), shuffle the six words, and ask a person which word does not belong.

- If people usually spot the intruder → the topic is coherent.
- If people guess wrong a lot → the topic words are mushy / overlapping.

Example of what one item looks like:

```text
topic 17: 1. gas  2. friday  3. pipeline  4. capacity  5. nomination  6. mmbtu
```

Here `friday` is the intruder (it came from some other topic). You would type `friday`.

## What this notebook does

1. Pull the three stage-4 winners from the latest finished HPO parent in MLflow.
2. Load their `*_model.pkl` files.
3. Build 30 intrusion items per finalist and **print them** (no hanging `input()`).
4. Optionally score your guesses from a dict you fill in.


In [1]:
from __future__ import annotations

import hashlib
import json
import random
from collections import defaultdict
from pathlib import Path

import mlflow
from bertopic import BERTopic
from IPython.display import clear_output

from email_cls_with_clustering.evaluate import word_intrusion
from email_cls_with_clustering.tracking import DEFAULT_EXPERIMENT, setup_tracking

NOTEBOOKS = Path(".").resolve()
if NOTEBOOKS.name != "notebooks":
    NOTEBOOKS = NOTEBOOKS / "notebooks"

TOPICS_PER_FINALIST = 30
JOB_NAME = "enron-email-topic-model-hpo"
STRATEGIES = ("c-tf-idf", "embeddings")


/home/marco/development/email-cls-with-clustering/.venv/lib/python3.13/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (1.26.13) or chardet (7.6.0)/charset_normalizer (2.1.1) doesn't match a supported version!
  warnings.warn(


In [2]:
def latest_hpo_parent(client: mlflow.tracking.MlflowClient, experiment_id: str):
    """Newest finished parent run that has stage-4 outlier children."""
    parents = client.search_runs(
        [experiment_id],
        filter_string=f"tags.mlflow.runName = '{JOB_NAME}'",
        order_by=["attributes.start_time DESC"],
        max_results=20,
    )
    for parent in parents:
        if parent.info.status != "FINISHED":
            continue
        children = client.search_runs(
            [experiment_id],
            filter_string=f"tags.mlflow.parentRunId = '{parent.info.run_id}'",
            max_results=5000,
        )
        stage4 = [c for c in children if c.data.params.get("outlier_strategy")]
        if stage4:
            return parent, children
    raise RuntimeError(
        f"No finished {JOB_NAME!r} parent with stage-4 outlier runs in MLflow."
    )


def parent_trial_id(run_name: str) -> str | None:
    for strategy in STRATEGIES:
        marker = f"__{strategy}__thr"
        if marker in run_name:
            return run_name.split(marker)[0]
    return None


def stage4_winners(children: list) -> list[dict]:
    """One winner per stage-3 finalist, matching hpo.run_stage4 selection."""
    stage3 = {
        r.info.run_name: r
        for r in children
        if not r.data.params.get("outlier_strategy") and "__mcs" in (r.info.run_name or "")
    }
    by_parent: dict[str, list] = defaultdict(list)
    for run in children:
        if not run.data.params.get("outlier_strategy"):
            continue
        pid = parent_trial_id(run.info.run_name or "")
        if pid is None:
            continue
        by_parent[pid].append(run)

    winners = []
    for pid, runs in sorted(by_parent.items()):
        baseline = stage3.get(pid)
        if baseline is None:
            print(f"skip {pid}: no stage-3 baseline in MLflow")
            continue
        b_noise = baseline.data.metrics.get("noise_share")
        b_coh = baseline.data.metrics.get("coherence_c_npmi")
        accepted = []
        for run in runs:
            if run.data.params.get("outlier_skipped") == "True":
                continue
            noise = run.data.metrics.get("noise_share")
            coh = run.data.metrics.get("coherence_c_npmi")
            if None in (noise, coh, b_noise, b_coh):
                continue
            if noise < b_noise and coh >= b_coh:
                accepted.append(run)
        if not accepted:
            print(f"skip {pid}: no accepted outlier reduction")
            continue
        accepted.sort(
            key=lambda r: (
                r.data.metrics["noise_share"],
                -r.data.metrics.get("coherence_c_npmi", float("-inf")),
            )
        )
        winner = accepted[0]
        winners.append(
            {
                "parent_trial_id": pid,
                "trial_id": winner.info.run_name,
                "run_id": winner.info.run_id,
                "noise_share": winner.data.metrics["noise_share"],
                "coherence_c_npmi": winner.data.metrics.get("coherence_c_npmi"),
                "n_topics": winner.data.metrics.get("n_topics"),
                "strategy": winner.data.params.get("outlier_strategy"),
                "threshold": winner.data.params.get("outlier_threshold"),
            }
        )
    return winners


def resolve_model_pkl(trial_id: str, notebooks: Path = NOTEBOOKS) -> Path:
    matches = sorted(notebooks.glob(f"emails_clustered_{trial_id}_*_model.pkl"))
    if not matches:
        raise FileNotFoundError(
            f"No model pickle for {trial_id!r} under {notebooks}. "
            "Stage 4 writes emails_clustered_<trial_id>_<timestamp>_model.pkl."
        )
    return matches[-1]


In [3]:
setup_tracking()
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(DEFAULT_EXPERIMENT)
parent, children = latest_hpo_parent(client, experiment.experiment_id)
winners = stage4_winners(children)

for row in winners:
    row["model_path"] = resolve_model_pkl(row["trial_id"])

print(f"parent run {parent.info.run_id} ({parent.info.status})")
print(f"found {len(winners)} stage-4 winners\n")
for row in winners:
    print(row["trial_id"])
    print(
        f"  noise={row['noise_share']:.4f}  "
        f"coherence={row['coherence_c_npmi']:.4f}  "
        f"n_topics={row['n_topics']:.0f}  "
        f"{row['strategy']}@{row['threshold']}"
    )
    print(f"  {row['model_path'].name}\n")

assert len(winners) >= 1, "expected at least one stage-4 winner"


parent run 735e206a162647558b93cd02a9098074 (FINISHED)
found 3 stage-4 winners

BAAI_bge-small-en-v1.5__mean_removal_plus_top_k__k2__umap_nc25_nn50__mcs50_ms15_eom_eps0.0__c-tf-idf__thr0.05
  noise=0.0453  coherence=0.1188  n_topics=553  c-tf-idf@0.05
  emails_clustered_BAAI_bge-small-en-v1.5__mean_removal_plus_top_k__k2__umap_nc25_nn50__mcs50_ms15_eom_eps0.0__c-tf-idf__thr0.05_20260926T223827_model.pkl

BAAI_bge-small-en-v1.5__mean_removal_plus_top_k__k2__umap_nc25_nn50__mcs50_ms1_eom_eps0.0__c-tf-idf__thr0.0
  noise=0.0006  coherence=0.1051  n_topics=715  c-tf-idf@0.0
  emails_clustered_BAAI_bge-small-en-v1.5__mean_removal_plus_top_k__k2__umap_nc25_nn50__mcs50_ms1_eom_eps0.0__c-tf-idf__thr0.0_20260926T214453_model.pkl

BAAI_bge-small-en-v1.5__mean_removal_plus_top_k__k2__umap_nc50_nn50__mcs50_ms5_eom_eps0.0__c-tf-idf__thr0.0
  noise=0.0005  coherence=0.1135  n_topics=628  c-tf-idf@0.0
  emails_clustered_BAAI_bge-small-en-v1.5__mean_removal_plus_top_k__k2__umap_nc50_nn50__mcs50_ms5_eo

In [4]:
models = {}
for row in winners:
    print(f"loading {row['model_path'].name} …")
    models[row["trial_id"]] = BERTopic.load(str(row["model_path"]))
print(f"loaded {len(models)} models")


loading emails_clustered_BAAI_bge-small-en-v1.5__mean_removal_plus_top_k__k2__umap_nc25_nn50__mcs50_ms15_eom_eps0.0__c-tf-idf__thr0.05_20260926T223827_model.pkl …
loading emails_clustered_BAAI_bge-small-en-v1.5__mean_removal_plus_top_k__k2__umap_nc25_nn50__mcs50_ms1_eom_eps0.0__c-tf-idf__thr0.0_20260926T214453_model.pkl …
loading emails_clustered_BAAI_bge-small-en-v1.5__mean_removal_plus_top_k__k2__umap_nc50_nn50__mcs50_ms5_eom_eps0.0__c-tf-idf__thr0.0_20260926T233241_model.pkl …
loaded 3 models


## Build intrusion items

For each winner we sample 30 topics. Each item is: 5 real top words + 1 intruder from another topic, shuffled.
Answers are stored but **not printed** in the quiz table so you can guess honestly.


In [5]:
def sample_topic_ids(model, n: int, seed: int) -> list[int]:
    ids = [topic_id for topic_id in model.get_topics() if topic_id != -1]
    if len(ids) < n:
        raise ValueError(f"model has {len(ids)} topics; need {n}")
    return random.Random(seed).sample(ids, n)


def make_intrusion_item(model, topic_id: int, seed: int, topk: int = 5) -> dict:
    """Same construction as evaluate.word_intrusion, but return structured data."""
    rng = random.Random(seed)
    lists = {
        t: [w for w, _ in words[:25]]
        for t, words in model.get_topics().items()
        if t != -1
    }
    real = lists[topic_id][:topk]
    pool = [
        w
        for t, ws in lists.items()
        if t != topic_id
        for w in ws[topk:]
        if w not in real
    ]
    if not pool:
        pool = [
            w
            for t, ws in lists.items()
            if t != topic_id
            for w in ws
            if w not in real
        ]
    intruder = rng.choice(pool)
    shown = real + [intruder]
    rng.shuffle(shown)
    return {
        "topic_id": topic_id,
        "shown": shown,
        "intruder": intruder,
        "real_words": real,
    }


quiz_plan = []
for row in winners:
    model = models[row["trial_id"]]
    seed = int(hashlib.md5(row["trial_id"].encode()).hexdigest()[:8], 16)
    topic_ids = sample_topic_ids(model, TOPICS_PER_FINALIST, seed=seed)
    items = [
        make_intrusion_item(model, topic_id, seed=topic_id + seed)
        for topic_id in topic_ids
    ]
    quiz_plan.append({**row, "topic_ids": topic_ids, "items": items})
    print(f"{row['trial_id']}")
    print(f"  {len(items)} items; first topic_id={items[0]['topic_id']} shown={items[0]['shown']}\n")


BAAI_bge-small-en-v1.5__mean_removal_plus_top_k__k2__umap_nc25_nn50__mcs50_ms15_eom_eps0.0__c-tf-idf__thr0.05
  30 items; first topic_id=440 shown=['variances', 'enrj', 'loadid', 'tblloads', 'met', 'detected']

BAAI_bge-small-en-v1.5__mean_removal_plus_top_k__k2__umap_nc25_nn50__mcs50_ms1_eom_eps0.0__c-tf-idf__thr0.0
  30 items; first topic_id=64 shown=['follows', 'gmt', 'attachment', 'undeliverable', 'inline', 'quotedprintable']

BAAI_bge-small-en-v1.5__mean_removal_plus_top_k__k2__umap_nc50_nn50__mcs50_ms5_eom_eps0.0__c-tf-idf__thr0.0
  30 items; first topic_id=385 shown=['page', 'labor', 'bna', 'sanseriffont', 'pw', 'bnaweb22']



## Quiz one finalist

Set `FINALIST_INDEX` to `0`, `1`, or `2`. The next cell **prints every item** as a numbered word list.
Fill `guesses` with the word you think is the intruder for each item index, then run the scoring cell.


In [6]:
FINALIST_INDEX = 0
plan = quiz_plan[FINALIST_INDEX]
print("finalist:", plan["trial_id"])
print(
    f"noise={plan['noise_share']:.4f}  "
    f"coherence={plan['coherence_c_npmi']:.4f}  "
    f"n_topics={plan['n_topics']:.0f}\n"
)
print("For each item, which ONE word does not belong with the others?\n")
for i, item in enumerate(plan["items"]):
    numbered = "  ".join(f"{j}. {w}" for j, w in enumerate(item["shown"], 1))
    print(f"[{i}] topic {item['topic_id']}: {numbered}")


finalist: BAAI_bge-small-en-v1.5__mean_removal_plus_top_k__k2__umap_nc25_nn50__mcs50_ms15_eom_eps0.0__c-tf-idf__thr0.05
noise=0.0453  coherence=0.1188  n_topics=553

For each item, which ONE word does not belong with the others?

[0] topic 440: 1. variances  2. enrj  3. loadid  4. tblloads  5. met  6. detected
[1] topic 65: 1. guaranty  2. guarantor  3. clement  4. password  5. obligations  6. form
[2] topic 54: 1. report  2. 01271997  3. weekly  4. monthly  5. bisttram  6. attached
[3] topic 437: 1. dj  2. caliso  3. hoffmanpdxect  4. awordaday  5. pwr  6. carla
[4] topic 89: 1. mou  2. assembly  3. bill  4. reichel  5. senate  6. sb
[5] topic 107: 1. pspan  2. alignleft  3. deal  4. classemailtextblack  5. wine  6. wines
[6] topic 133: 1. caps  2. california  3. migdenna  4. cap  5. ferc  6. iso
[7] topic 416: 1. tel  2. houston  3. mmbtus  4. ste  5. family  6. tx
[8] topic 371: 1. azurix  2. rab  3. dependence  4. azurixs  5. water  6. wessex
[9] topic 70: 1. approval  2. request  

In [8]:
# Map item index -> the word you think is the intruder.
# Example: guesses = {0: "friday", 1: "meeting", ...}
guesses: dict[int, str] = {0: "enrj", 1: "password", 2: "01271997"}

results = []
for i, item in enumerate(plan["items"]):
    guess = guesses.get(i)
    answer = item["intruder"]
    correct = guess is not None and guess.lower() == answer.lower()
    results.append(
        {
            "index": i,
            "topic_id": item["topic_id"],
            "shown": item["shown"],
            "guess": guess,
            "answer": answer,
            "correct": correct,
        }
    )

answered = [r for r in results if r["guess"] is not None]
if not answered:
    print("No guesses yet. Fill `guesses` above, then re-run this cell.")
    print("Reveal answers for the first 3 items as a sanity check:\n")
    for r in results[:3]:
        print(f"[{r['index']}] shown={r['shown']}  intruder={r['answer']!r}")
else:
    hits = sum(r["correct"] for r in answered)
    print(f"score: {hits}/{len(answered)} ({hits / len(answered):.0%}) on answered items")
    for r in answered:
        mark = "OK" if r["correct"] else f"NO (answer={r['answer']!r})"
        print(f"[{r['index']}] guess={r['guess']!r}  {mark}")

out = NOTEBOOKS / f"stage5_word_intrusion_{plan['trial_id']}.json"
payload = {
    "parent_run_id": parent.info.run_id,
    "trial_id": plan["trial_id"],
    "model_path": str(plan["model_path"]),
    "metrics": {
        "noise_share": plan["noise_share"],
        "coherence_c_npmi": plan["coherence_c_npmi"],
        "n_topics": plan["n_topics"],
    },
    "results": results,
    "n_answered": len(answered),
    "score": (sum(r["correct"] for r in answered) / len(answered)) if answered else None,
}
out.write_text(json.dumps(payload, indent=2) + "\n")
print("\nwrote", out)


score: 1/3 (33%) on answered items
[0] guess='enrj'  NO (answer='met')
[1] guess='password'  OK
[2] guess='01271997'  NO (answer='bisttram')

wrote /home/marco/development/email-cls-with-clustering/notebooks/stage5_word_intrusion_BAAI_bge-small-en-v1.5__mean_removal_plus_top_k__k2__umap_nc25_nn50__mcs50_ms15_eom_eps0.0__c-tf-idf__thr0.05.json
